# Baitarani hourly data: GUARDIAN dataset + CWC live levels

No GPU needed: set **Accelerator: None** and **Internet: On**. Takes a few minutes, so you can run the cells normally (no need for Save & Run All). Results go to the **Output** tab under `results/guardian_baitarani/`.

1. GUARDIAN (Scientific Data 2024): hourly monsoon water level / discharge for 210 CWC stations from 2020 — are Anandpur and Akhuapada included?
2. HydroSwift: list the CWC flood-forecasting stations on the Baitarani and download recent water levels.

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.isdir('flood'):
    subprocess.run('git clone -q https://github.com/yatharth7115/flood.git', shell=True, check=True)
os.chdir('/kaggle/working/flood')
subprocess.run('git pull -q && git log --oneline -1', shell=True)
os.makedirs('/kaggle/working/results/guardian_baitarani', exist_ok=True)

## 1. GUARDIAN: download and search for Baitarani stations

In [ ]:
!python src/guardian_baitarani.py /tmp/guardian /kaggle/working/results/guardian_baitarani || echo 'GUARDIAN step failed - continuing'

Only if the cell above says *Not downloaded from Figshare*: add the files as a Kaggle dataset from the Figshare link (see the message above), then run this cell.

In [ ]:
import glob
inp = sorted(glob.glob('/kaggle/input/*guardian*'))
print(inp)
if inp:
    !python src/guardian_baitarani.py /tmp/guardian2 /kaggle/working/results/guardian_baitarani {inp[0]}


## 2. CWC flood-forecasting stations on the Baitarani (HydroSwift)
If this cell fails, copy the error; the package API may differ from the documentation.

In [ ]:
# HydroSwift is not on PyPI: install it from its GitHub repo
!pip install -q git+https://github.com/carbform/HydroSwift.git
import hydroswift, pandas as pd
print('HydroSwift', getattr(hydroswift, '__version__', '?'))
st = None
for kw in [dict(river='Baitarani'), dict(river='Baitarni'), {}]:
    try:
        st = hydroswift.cwc.stations(**kw)
        if not kw:
            txt = st.astype(str).apply(lambda c: c.str.lower())
            st = st[txt.apply(lambda c: c.str.contains('baitar|anandpur|akhuapada|champua|swampatna')).any(axis=1)]
        print(kw or 'all stations, filtered by name', '->', len(st), 'stations')
        if len(st):
            break
    except Exception as e:
        print(kw, 'failed:', e)
if st is not None and len(st):
    print(st.to_string()[:4000])
    st.to_csv('/kaggle/working/results/guardian_baitarani/cwc_baitarani_stations.csv', index=False)
else:
    print('No Baitarani CWC stations found')


In [ ]:
# recent water levels for the Baitarani stations (the CWC portal keeps only recent data)
try:
    out = hydroswift.fetch(st, output_dir='/kaggle/working/results/guardian_baitarani/cwc_live', merge=True)
    print(out.head(20) if hasattr(out, 'head') else out)
except Exception as e:
    print('download failed:', e)
!ls -R /kaggle/working/results/guardian_baitarani | head -50